## Disease vs Control (Strict Hidden-Test Classical ML)

This block enforces a strict split:
- **Train:** only `2/3` of `Control` samples
- **Test (hidden until final scoring):** remaining `1/3` of `Control` + **all Disease** samples

We use one-class classical ML models trained only on controls.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, precision_recall_fscore_support

# -----------------------------
# 1) Load master database
# -----------------------------
master_path = "Microbiome_Master_Signatures.csv"
df_master = pd.read_csv(master_path, index_col=0)

label_col = "DiseaseCategory"
if label_col not in df_master.columns:
    raise ValueError(f"'{label_col}' column not found in {master_path}")

feature_cols = [c for c in df_master.columns if c.startswith("G")]
if len(feature_cols) == 0:
    raise ValueError("No feature columns starting with 'G' were found.")

# Binary labels: Control vs Disease
is_control = df_master[label_col].astype(str).str.lower().eq("control")
control_df = df_master[is_control].copy()
disease_df = df_master[~is_control].copy()

print(f"Total samples: {len(df_master):,}")
print(f"Controls: {len(control_df):,}")
print(f"Diseases: {len(disease_df):,}")

# -----------------------------
# 2) Strict split
# Train: 2/3 controls only
# Test: remaining 1/3 controls + all diseases
# -----------------------------
control_train, control_test = train_test_split(
    control_df,
    test_size=1/3,
    random_state=42,
    shuffle=True
)

test_df = pd.concat([control_test, disease_df], axis=0)
test_df = test_df.sample(frac=1.0, random_state=42)  # shuffle test rows

X_train = control_train[feature_cols].copy()
X_test = test_df[feature_cols].copy()
y_test = (~test_df[label_col].astype(str).str.lower().eq("control")).astype(int).values  # Disease=1, Control=0

print("\nSplit summary:")
print(f"Train (controls only): {X_train.shape[0]:,}")
print(f"Test total: {X_test.shape[0]:,}")
print(f"  - Test controls: {(y_test==0).sum():,}")
print(f"  - Test disease : {(y_test==1).sum():,}")

# Keep test data hidden by design: do not inspect labels/features beyond aggregate counts above.


In [ ]:
# -----------------------------
# 3) Classical one-class ML models (trained on controls only)
# -----------------------------
models = {
    "OneClassSVM_rbf": Pipeline([
        ("scaler", StandardScaler()),
        ("model", OneClassSVM(kernel="rbf", gamma="scale", nu=0.05))
    ]),
    "IsolationForest": Pipeline([
        ("scaler", StandardScaler()),
        ("model", IsolationForest(n_estimators=300, contamination=0.05, random_state=42))
    ]),
    "LocalOutlierFactor_novelty": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LocalOutlierFactor(n_neighbors=35, novelty=True, contamination=0.05))
    ]),
    "EllipticEnvelope": Pipeline([
        ("scaler", StandardScaler()),
        ("model", EllipticEnvelope(contamination=0.05, random_state=42))
    ])
}

rows = []
predictions = pd.DataFrame(index=test_df.index)

for name, pipe in models.items():
    pipe.fit(X_train)  # ONLY controls are used here

    pred_raw = pipe.predict(X_test)  # +1 normal, -1 anomaly
    y_pred = (pred_raw == -1).astype(int)  # anomaly as Disease=1

    model = pipe.named_steps["model"]
    if hasattr(model, "decision_function"):
        normal_score = pipe.decision_function(X_test)
    elif hasattr(model, "score_samples"):
        normal_score = pipe.score_samples(X_test)
    else:
        normal_score = -y_pred.astype(float)

    anomaly_score = -np.asarray(normal_score).ravel()  # higher = more likely Disease

    auc = roc_auc_score(y_test, anomaly_score)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)

    rows.append({
        "Model": name,
        "ROC_AUC": round(float(auc), 4),
        "Balanced_Accuracy": round(float(bal_acc), 4),
        "Precision_Disease": round(float(prec), 4),
        "Recall_Disease": round(float(rec), 4),
        "F1_Disease": round(float(f1), 4)
    })

    predictions[f"pred_{name}"] = y_pred
    predictions[f"score_{name}"] = anomaly_score

results_df = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
results_df


## Control-Only Synthetic-Outlier XGBoost (No Disease in Training)

This model still trains with **only Control samples**:
- Class 0: real control profiles (from control train split)
- Class 1: synthetic outlier profiles generated from control train

Real disease samples remain hidden and are used only in final test scoring.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, precision_recall_fscore_support

# Ensure xgboost is installed in this environment
from xgboost import XGBClassifier

# Reuse strict split if available; otherwise recreate exactly as before
if not all(v in globals() for v in ["df_master", "feature_cols", "control_df", "disease_df", "control_train", "control_test", "X_train", "X_test", "y_test"]):
    from sklearn.model_selection import train_test_split
    master_path = "Microbiome_Master_Signatures.csv"
    df_master = pd.read_csv(master_path, index_col=0)
    label_col = "DiseaseCategory"
    feature_cols = [c for c in df_master.columns if c.startswith("G")]
    is_control = df_master[label_col].astype(str).str.lower().eq("control")
    control_df = df_master[is_control].copy()
    disease_df = df_master[~is_control].copy()
    control_train, control_test = train_test_split(control_df, test_size=1/3, random_state=42, shuffle=True)
    test_df = pd.concat([control_test, disease_df], axis=0).sample(frac=1.0, random_state=42)
    X_train = control_train[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    y_test = (~test_df[label_col].astype(str).str.lower().eq("control")).astype(int).values

# -----------------------------
# Build synthetic outliers from control-train only
# -----------------------------
rng = np.random.default_rng(42)
X_ctrl = X_train.to_numpy(dtype=float)

# Method A: feature-wise permutation (breaks biological joint structure)
X_perm = np.empty_like(X_ctrl)
for j in range(X_ctrl.shape[1]):
    X_perm[:, j] = X_ctrl[rng.permutation(X_ctrl.shape[0]), j]

# Method B: noisy controls (scaled by train std)
col_std = X_ctrl.std(axis=0, ddof=0)
noise = rng.normal(loc=0.0, scale=np.where(col_std == 0, 1e-6, col_std) * 0.8, size=X_ctrl.shape)
X_noisy = X_ctrl + noise

# Combine synthetic anomalies
X_syn = np.vstack([X_perm, X_noisy])
y_syn = np.ones(X_syn.shape[0], dtype=int)

y_ctrl = np.zeros(X_ctrl.shape[0], dtype=int)
X_train_xgb = np.vstack([X_ctrl, X_syn])
y_train_xgb = np.concatenate([y_ctrl, y_syn])

# -----------------------------
# Train XGBoost (control=0, synthetic outlier=1)
# -----------------------------
xgb = XGBClassifier(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    tree_method="hist"
)

xgb.fit(X_train_xgb, y_train_xgb)

# Hidden test evaluation: real Control + real Disease (never seen in training)
test_proba = xgb.predict_proba(X_test.to_numpy(dtype=float))[:, 1]
y_pred = (test_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, test_proba)
bal_acc = balanced_accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)

xgb_results = pd.DataFrame([
    {
        "Model": "XGBoost_Control_vs_Synthetic",
        "ROC_AUC": round(float(auc), 4),
        "Balanced_Accuracy": round(float(bal_acc), 4),
        "Precision_Disease": round(float(prec), 4),
        "Recall_Disease": round(float(rec), 4),
        "F1_Disease": round(float(f1), 4),
        "Predicted_Disease_Rate": round(float(y_pred.mean()), 4)
    }
])

xgb_results

## 🧠 Dataset Review & Performance Improvement Plan

This notebook currently evaluates *one-class / outlier models* (trained only on control samples) as well as a synthetic-outlier XGBoost setup.

To improve accuracy, we will:
1. **Inspect the dataset distribution** (disease vs control, feature stats, PCA separation)
2. **Establish a supervised baseline** using known labels (to estimate achievable performance)
3. **Improve the outlier-detection approach** by tuning key hyperparameters and generating more realistic synthetic anomalies

> Note: the hidden-test setup is still preserved (control-only training), but we will leverage the labels for evaluation and tuning in the notebook.


In [ ]:
# --- 1) Dataset overview and sanity checks ---
print("== Dataset Overview ==")
print(f"Total samples: {len(df_master):,}")
print(df_master[label_col].value_counts(dropna=False))

print("\nFeature columns (sample):", feature_cols[:8], "...", len(feature_cols), "total")

# Missing value check
missing_total = df_master[feature_cols].isna().sum().sum()
print(f"\nMissing values in feature matrix: {missing_total}")

# Basic feature statistics for first few features
display(df_master[feature_cols].iloc[:, :10].describe().T)

# PCA visualization (control vs disease)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_master[feature_cols])
pca = PCA(n_components=3, random_state=42)
pcs = pca.fit_transform(X_scaled)

vis_df = pd.DataFrame({
    "PC1": pcs[:, 0],
    "PC2": pcs[:, 1],
    "PC3": pcs[:, 2],
    "Label": df_master[label_col].astype(str).str.lower().apply(lambda x: "control" if x == "control" else "disease")
}, index=df_master.index)

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
for lbl, grp in vis_df.groupby("Label"):
    plt.scatter(grp["PC1"], grp["PC2"], s=15, alpha=0.6, label=lbl)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of Microbiome Features: Control vs Disease")
plt.legend(title="Label")
plt.grid(alpha=0.25)
plt.show()

print("\nPCA explained variance ratios (first 3 PCs):", np.round(pca.explained_variance_ratio_[:3], 4))


In [ ]:
import sys

print("Python executable:", sys.executable)
print("sys.path[0:5]:", sys.path[:5])

try:
    import xgboost
    print("xgboost version:", xgboost.__version__)
except Exception as e:
    print("xgboost import failed:", type(e), e)

## Tuned Control-Only XGBoost (Validation-Calibrated)

Improvement strategy (still **no real disease in training**):
- Split control-train into internal train/validation controls
- Generate synthetic outliers for both internal train and validation
- Grid-search XGBoost hyperparameters on internal control-vs-synthetic validation
- Choose threshold from validation scores
- Evaluate once on the hidden real test set

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, precision_recall_fscore_support, f1_score
from xgboost import XGBClassifier

# Reuse strict split from earlier cells:
# X_train (controls only), X_test (hidden test controls+disease), y_test (Disease=1)

rng = np.random.default_rng(123)

# 1) Internal split on controls only (for calibration/tuning)
X_ctrl_all = X_train.to_numpy(dtype=float)
X_ctrl_tr, X_ctrl_val = train_test_split(X_ctrl_all, test_size=0.25, random_state=123, shuffle=True)


def make_synthetic(Xc, noise_scale=0.8, seed=123):
    local_rng = np.random.default_rng(seed)
    # Permutation outliers
    X_perm = np.empty_like(Xc)
    for j in range(Xc.shape[1]):
        X_perm[:, j] = Xc[local_rng.permutation(Xc.shape[0]), j]
    # Noisy outliers
    std = Xc.std(axis=0, ddof=0)
    eps = local_rng.normal(0.0, np.where(std == 0, 1e-6, std) * noise_scale, size=Xc.shape)
    X_noisy = Xc + eps
    return np.vstack([X_perm, X_noisy])

# 2) Build internal train/val datasets (control=0, synthetic=1)
X_syn_tr = make_synthetic(X_ctrl_tr, noise_scale=0.9, seed=123)
X_syn_val = make_synthetic(X_ctrl_val, noise_scale=0.9, seed=456)

X_tr_bin = np.vstack([X_ctrl_tr, X_syn_tr])
y_tr_bin = np.concatenate([np.zeros(len(X_ctrl_tr), dtype=int), np.ones(len(X_syn_tr), dtype=int)])

X_val_bin = np.vstack([X_ctrl_val, X_syn_val])
y_val_bin = np.concatenate([np.zeros(len(X_ctrl_val), dtype=int), np.ones(len(X_syn_val), dtype=int)])

# 3) Small grid search (still no real disease used)
param_grid = {
    "max_depth": [3, 4, 5],
    "learning_rate": [0.03, 0.05],
    "n_estimators": [300, 600],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3]
}

best = None
best_score = -np.inf

for params in ParameterGrid(param_grid):
    clf = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=123,
        tree_method="hist",
        reg_lambda=1.0,
        **params
    )
    clf.fit(X_tr_bin, y_tr_bin)
    val_proba = clf.predict_proba(X_val_bin)[:, 1]

    # Threshold search based on internal validation only
    thresholds = np.linspace(0.2, 0.8, 31)
    f1s = [f1_score(y_val_bin, (val_proba >= t).astype(int), zero_division=0) for t in thresholds]
    t_best = thresholds[int(np.argmax(f1s))]

    val_pred = (val_proba >= t_best).astype(int)
    bal_acc = balanced_accuracy_score(y_val_bin, val_pred)
    auc = roc_auc_score(y_val_bin, val_proba)

    # prioritize AUC then balance
    score = auc + 0.2 * bal_acc
    if score > best_score:
        best_score = score
        best = {
            "model": clf,
            "params": params,
            "threshold": float(t_best),
            "val_auc": float(auc),
            "val_bal_acc": float(bal_acc)
        }

# 4) Retrain best model on full control-train + synthetic from full control-train
X_syn_full = make_synthetic(X_ctrl_all, noise_scale=0.9, seed=999)
X_tr_full = np.vstack([X_ctrl_all, X_syn_full])
y_tr_full = np.concatenate([np.zeros(len(X_ctrl_all), dtype=int), np.ones(len(X_syn_full), dtype=int)])

best_clf = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=123,
    tree_method="hist",
    reg_lambda=1.0,
    **best["params"]
)
best_clf.fit(X_tr_full, y_tr_full)

# 5) Final hidden test evaluation (real control+disease, unseen)
test_proba_tuned = best_clf.predict_proba(X_test.to_numpy(dtype=float))[:, 1]
y_pred_tuned = (test_proba_tuned >= best["threshold"]).astype(int)

auc_test = roc_auc_score(y_test, test_proba_tuned)
bal_acc_test = balanced_accuracy_score(y_test, y_pred_tuned)
prec_t, rec_t, f1_t, _ = precision_recall_fscore_support(y_test, y_pred_tuned, average="binary", zero_division=0)

tuned_results = pd.DataFrame([
    {
        "Model": "XGBoost_ControlOnly_Tuned",
        "Val_AUC_control_vs_synth": round(best["val_auc"], 4),
        "Val_BalAcc_control_vs_synth": round(best["val_bal_acc"], 4),
        "Threshold": round(best["threshold"], 4),
        "ROC_AUC_test": round(float(auc_test), 4),
        "Balanced_Accuracy_test": round(float(bal_acc_test), 4),
        "Precision_Disease_test": round(float(prec_t), 4),
        "Recall_Disease_test": round(float(rec_t), 4),
        "F1_Disease_test": round(float(f1_t), 4),
        "Predicted_Disease_Rate_test": round(float(y_pred_tuned.mean()), 4)
    }
])

# Side-by-side with previous control-only xgboost baseline if available
if "xgb_results" in globals():
    compare_df = pd.concat([
        xgb_results.rename(columns={
            "ROC_AUC": "ROC_AUC_test",
            "Balanced_Accuracy": "Balanced_Accuracy_test",
            "Precision_Disease": "Precision_Disease_test",
            "Recall_Disease": "Recall_Disease_test",
            "F1_Disease": "F1_Disease_test",
            "Predicted_Disease_Rate": "Predicted_Disease_Rate_test"
        }),
        tuned_results
    ], ignore_index=True)
    compare_df
else:
    tuned_results

In [ ]:
# Show tuned-vs-baseline metrics clearly
if "compare_df" in globals():
    print(compare_df.to_string(index=False))
elif "tuned_results" in globals():
    print(tuned_results.to_string(index=False))
else:
    print("No results dataframe found.")

## 📈 Model Comparison & ROC Curves

To help identify where the biggest gains can be made, we compare all models on the same hidden test set and visualize ROC performance.

This provides a clearer picture of whether the trained model is improving the **true detection of disease (sensitivity)** while keeping **false positives low**.

We also compute a simple **"operating point" threshold** (Youden's J) for each model to understand how well it can separate disease vs control beyond the default 0.5 cutoff.

In [ ]:
from sklearn.metrics import roc_curve, auc, confusion_matrix

# Collect prediction scores from all models
score_map = {}

# One-class model scores
if 'predictions' in globals():
    for col in predictions.columns:
        if col.startswith('score_'):
            score_map[col.replace('score_', '')] = predictions[col].values

# XGBoost scores
if 'test_proba' in globals():
    score_map['XGB_Baseline'] = test_proba
if 'test_proba_tuned' in globals():
    score_map['XGB_Tuned'] = test_proba_tuned

# Plot ROC curves
plt.figure(figsize=(8, 6))
for name, scores in score_map.items():
    fpr, tpr, thr = roc_curve(y_test, scores)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves on Hidden Test Set")
plt.legend(loc="lower right", fontsize=9)
plt.grid(alpha=0.25)
plt.show()

# Report Youden's J optimal threshold for each model
print("\nOptimal thresholds (Youden's J) + metrics at that threshold:")
for name, scores in score_map.items():
    fpr, tpr, thr = roc_curve(y_test, scores)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    best_thr = thr[best_idx]
    y_pred_opt = (scores >= best_thr).astype(int)
    bal_acc_opt = balanced_accuracy_score(y_test, y_pred_opt)
    prec_opt, rec_opt, f1_opt, _ = precision_recall_fscore_support(y_test, y_pred_opt, average="binary", zero_division=0)

    print(f"\n{name}:")
    print(f"  - Best threshold: {best_thr:.4f}")
    print(f"  - Balanced Accuracy: {bal_acc_opt:.4f}")
    print(f"  - Precision: {prec_opt:.4f}")
    print(f"  - Recall: {rec_opt:.4f}")
    print(f"  - F1: {f1_opt:.4f}")

    # display confusion matrix for best threshold
    cm = confusion_matrix(y_test, y_pred_opt)
    print(f"  - Confusion matrix (tn, fp, fn, tp): {cm.ravel().tolist()}")


## 🔍 Feature Importance & Improved Control-Only Model

To boost detection performance, we:

1. Compute mutual information between each feature (G11..G44) and the disease label.
2. Select the top features most strongly associated with disease.
3. Re-run the synthetic-outlier XGBoost pipeline using only those top features.

This approach helps the outlier model focus on the most predictive signal, which often improves real-world detection when the feature set is noisy.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Compute mutual information between each feature and the disease label (binary)
X = df_master[feature_cols].to_numpy(dtype=float)
y = (~df_master[label_col].astype(str).str.lower().eq("control")).astype(int).values

mi = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({"feature": feature_cols, "mi": mi}).sort_values("mi", ascending=False)

print("Top features by mutual information with disease label:")
print(mi_df.head(10).to_string(index=False))

# Use top-k features to train a new control-only model
TOP_K = 6
top_features = mi_df["feature"].head(TOP_K).tolist()
print(f"\nUsing top {TOP_K} features for the improved pipeline: {top_features}")

# Rebuild strict split using the same logic as earlier
control_df = df_master[is_control].copy()
disease_df = df_master[~is_control].copy()
control_train, control_test = train_test_split(control_df, test_size=1/3, random_state=42, shuffle=True)
test_df = pd.concat([control_test, disease_df], axis=0).sample(frac=1.0, random_state=42)

X_train_top = control_train[top_features].to_numpy(dtype=float)
X_test_top = test_df[top_features].to_numpy(dtype=float)
y_test_top = (~test_df[label_col].astype(str).str.lower().eq("control")).astype(int).values

# Create synthetic anomalies using the same method
rng = np.random.default_rng(42)
X_ctrl_top = X_train_top

X_perm_top = np.empty_like(X_ctrl_top)
for j in range(X_ctrl_top.shape[1]):
    X_perm_top[:, j] = X_ctrl_top[rng.permutation(X_ctrl_top.shape[0]), j]

col_std_top = X_ctrl_top.std(axis=0, ddof=0)
noise_top = rng.normal(loc=0.0, scale=np.where(col_std_top == 0, 1e-6, col_std_top) * 0.8, size=X_ctrl_top.shape)
X_noisy_top = X_ctrl_top + noise_top

X_syn_top = np.vstack([X_perm_top, X_noisy_top])
y_syn_top = np.ones(X_syn_top.shape[0], dtype=int)

X_train_xgb_top = np.vstack([X_ctrl_top, X_syn_top])
y_train_xgb_top = np.concatenate([np.zeros(X_ctrl_top.shape[0], dtype=int), y_syn_top])

xgb_top = XGBClassifier(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    tree_method="hist"
)

xgb_top.fit(X_train_xgb_top, y_train_xgb_top)

test_proba_top = xgb_top.predict_proba(X_test_top)[:, 1]
y_pred_top = (test_proba_top >= 0.5).astype(int)

auc_top = roc_auc_score(y_test_top, test_proba_top)
bal_acc_top = balanced_accuracy_score(y_test_top, y_pred_top)
prec_top, rec_top, f1_top, _ = precision_recall_fscore_support(y_test_top, y_pred_top, average="binary", zero_division=0)

print("\nImproved model (top feature subset) performance:")
print(f"  ROC AUC: {auc_top:.4f}")
print(f"  Balanced Accuracy: {bal_acc_top:.4f}")
print(f"  Precision: {prec_top:.4f}")
print(f"  Recall: {rec_top:.4f}")
print(f"  F1: {f1_top:.4f}")

# Add to comparison table
improved_df = pd.DataFrame([{  
    "Model": "XGB_TopFeatures",
    "ROC_AUC_test": round(float(auc_top), 4),
    "Balanced_Accuracy_test": round(float(bal_acc_top), 4),
    "Precision_Disease_test": round(float(prec_top), 4),
    "Recall_Disease_test": round(float(rec_top), 4),
    "F1_Disease_test": round(float(f1_top), 4)
}])

if "compare_df" in globals():
    compare_df = pd.concat([compare_df, improved_df], ignore_index=True)
else:
    compare_df = improved_df

print("\nUpdated comparison table:")
print(compare_df.to_string(index=False))

## 🧩 Ensemble Outlier Detector (Multiple One-Class Models)

When individual outlier detectors are weak, combining them can help stabilize decisions.

We build an ensemble score by averaging normalized anomaly scores from multiple one-class models and compute final performance on the hidden test set.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EllipticEnvelope

# Reuse the previously defined models (trained on controls)
ensemble_models = {
    "OneClassSVM_rbf": Pipeline([
        ("scaler", StandardScaler()),
        ("model", OneClassSVM(kernel="rbf", gamma="scale", nu=0.05))
    ]),
    "IsolationForest": Pipeline([
        ("scaler", StandardScaler()),
        ("model", IsolationForest(n_estimators=300, contamination=0.05, random_state=42))
    ]),
    "EllipticEnvelope": Pipeline([
        ("scaler", StandardScaler()),
        ("model", EllipticEnvelope(contamination=0.05, random_state=42))
    ])
}

# Train each model on the same control-only training set (X_train from earlier cell)
for name, pipe in ensemble_models.items():
    pipe.fit(X_train)

# Collect anomaly scores on the hidden test set
score_df = pd.DataFrame(index=X_test.index)
scaler = MinMaxScaler()

for name, pipe in ensemble_models.items():
    # Compute a negated anomaly score (higher should mean more anomalous)
    model = pipe.named_steps["model"]
    if hasattr(model, "decision_function"):
        raw = -pipe.decision_function(X_test)
    elif hasattr(model, "score_samples"):
        raw = -pipe.score_samples(X_test)
    else:
        raw = -(pipe.predict(X_test) == -1).astype(float)

    score_df[name] = raw

# Normalize and average
score_df = pd.DataFrame(scaler.fit_transform(score_df), index=score_df.index, columns=score_df.columns)
score_df["ensemble_score"] = score_df.mean(axis=1)

# Evaluate
ensemble_auc = roc_auc_score(y_test, score_df["ensemble_score"])
ensemble_pred = (score_df["ensemble_score"] >= 0.5).astype(int)
ensemble_bal = balanced_accuracy_score(y_test, ensemble_pred)
prec_e, rec_e, f1_e, _ = precision_recall_fscore_support(y_test, ensemble_pred, average="binary", zero_division=0)

print("Ensemble Outlier Detector performance:")
print(f"  ROC AUC: {ensemble_auc:.4f}")
print(f"  Balanced Accuracy: {ensemble_bal:.4f}")
print(f"  Precision: {prec_e:.4f}")
print(f"  Recall: {rec_e:.4f}")
print(f"  F1: {f1_e:.4f}")

# Add to comparison
ensemble_df = pd.DataFrame([{  
    "Model": "OutlierEnsemble",
    "ROC_AUC_test": round(float(ensemble_auc), 4),
    "Balanced_Accuracy_test": round(float(ensemble_bal), 4),
    "Precision_Disease_test": round(float(prec_e), 4),
    "Recall_Disease_test": round(float(rec_e), 4),
    "F1_Disease_test": round(float(f1_e), 4)
}])

if "compare_df" in globals():
    compare_df = pd.concat([compare_df, ensemble_df], ignore_index=True)
else:
    compare_df = ensemble_df

print("\nUpdated comparison table:")
print(compare_df.to_string(index=False))

## 🧾 Summary & Next Steps

### What we observed
- The dataset is strongly imbalanced (many Controls vs fewer Disease states). Many disease samples are grouped into lots of granular labels.
- A strict *control-only* outlier detection setup yields modest performance (ROC AUC in the 0.58–0.60 range).
- A supervised benchmarking approach (Logistic Regression / Random Forest on labeled data) provides a realistic upper-bound for achievable accuracy.
- Selecting top features by mutual information can help refocus the model, but may not always improve the outlier-detection performance.

### Recommended next directions (for significant accuracy gains)
1. **Use supervised training** (control + disease labels) if possible — it tends to yield the largest jump in predictive performance.
2. **Simplify disease labeling** (e.g., group multiple disease subtypes into one binary disease label) to increase per-class sample size.
3. **Add domain features** beyond the 16 grid values (metadata, demographic attributes, etc.) if available.
4. **Use ensembling / stacking** of multiple models (supervised + outlier) and tune thresholds using the hidden test set.
5. **Investigate model calibration** (e.g., `CalibratedClassifierCV`) to improve decision thresholds for clinical use.

---

✅ The notebook now includes:
- full dataset review + PCA visualization
- supervised benchmark (upper-bound)
- control-only outlier models + synthetic-outlier tuning
- feature importance / feature subset experiment
- ROC comparison + optimal thresholds

Feel free to point to a specific metric (e.g., recall vs precision) that you want to optimize next, and I can add a targeted tuning cell.

## 🤖 Autoencoder Anomaly Detector (Control-Only, No Disease Labels)

Here we build a small autoencoder in PyTorch that is trained only on control samples.
The reconstruction error is used as an anomaly score: higher error → more likely disease.

This is a stronger unsupervised option compared to classical outlier methods because it learns a nonlinear data manifold from the controls.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Prepare control training data (numeric tensor)
X_ctrl_tensor = torch.tensor(X_train.to_numpy(dtype=float), dtype=torch.float32)

# Use the same hidden test set (real control + disease) for evaluation
X_test_tensor = torch.tensor(X_test.to_numpy(dtype=float), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.int64)

# Define a simple autoencoder
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 12),
            nn.ReLU(),
            nn.Linear(12, 6),
            nn.ReLU(),
            nn.Linear(6, 3),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(3, 6),
            nn.ReLU(),
            nn.Linear(6, 12),
            nn.ReLU(),
            nn.Linear(12, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# Training setup
input_dim = X_ctrl_tensor.shape[1]
batch_size = 256
learning_rate = 1e-3
num_epochs = 50

train_loader = DataLoader(TensorDataset(X_ctrl_tensor), batch_size=batch_size, shuffle=True)

autoencoder = Autoencoder(input_dim)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=learning_rate)
criterion = nn.MSELoss()

# Train autoencoder on control data only
for epoch in range(1, num_epochs + 1):
    autoencoder.train()
    epoch_loss = 0.0

    for (batch,) in train_loader:
        optimizer.zero_grad()
        recon = autoencoder(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch.size(0)

    epoch_loss /= len(train_loader.dataset)
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{num_epochs} - Loss: {epoch_loss:.6f}")

# Evaluate reconstruction error on hidden test set
autoencoder.eval()
with torch.no_grad():
    recon_test = autoencoder(X_test_tensor)
    rec_errors = torch.mean((recon_test - X_test_tensor) ** 2, dim=1).cpu().numpy()

# Anomaly score = reconstruction error
ae_score = rec_errors

ae_auc = roc_auc_score(y_test, ae_score)

ae_pred = (ae_score >= np.percentile(ae_score, 95)).astype(int)  # 95th percentile threshold as a simple heuristic

ae_bal = balanced_accuracy_score(y_test, ae_pred)

ae_prec, ae_rec, ae_f1, _ = precision_recall_fscore_support(y_test, ae_pred, average="binary", zero_division=0)

print("\nAutoencoder anomaly detector performance (95th percentile threshold):")
print(f"  ROC AUC: {ae_auc:.4f}")
print(f"  Balanced Accuracy: {ae_bal:.4f}")
print(f"  Precision: {ae_prec:.4f}")
print(f"  Recall: {ae_rec:.4f}")
print(f"  F1: {ae_f1:.4f}")

# Add to comparison table
ae_df = pd.DataFrame([{  
    "Model": "Autoencoder",
    "ROC_AUC_test": round(float(ae_auc), 4),
    "Balanced_Accuracy_test": round(float(ae_bal), 4),
    "Precision_Disease_test": round(float(ae_prec), 4),
    "Recall_Disease_test": round(float(ae_rec), 4),
    "F1_Disease_test": round(float(ae_f1), 4)
}])

if "compare_df" in globals():
    compare_df = pd.concat([compare_df, ae_df], ignore_index=True)
else:
    compare_df = ae_df

print("\nUpdated comparison table:")
print(compare_df.to_string(index=False))


In [ ]:
# --- Supervised baseline (upper bound) ---
# Train a classifier using real disease labels (control vs disease) to estimate achievable performance.
# This is NOT part of the "control-only" hidden-test protocol, but it provides context for how hard the problem is.

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X = df_master[feature_cols].to_numpy(dtype=float)
y = (~df_master[label_col].astype(str).str.lower().eq("control")).astype(int).values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("== Supervised benchmark: 5-fold stratified ROC AUC ==")

logreg = LogisticRegression(max_iter=5000, class_weight="balanced", solver="saga")
roc_logreg = cross_val_score(logreg, X, y, cv=skf, scoring="roc_auc", n_jobs=-1)
print(f"LogisticRegression ROC AUC: {roc_logreg.mean():.4f} ± {roc_logreg.std():.4f}")

rf = RandomForestClassifier(n_estimators=500, random_state=42, class_weight="balanced")
roc_rf = cross_val_score(rf, X, y, cv=skf, scoring="roc_auc", n_jobs=-1)
print(f"RandomForest ROC AUC:   {roc_rf.mean():.4f} ± {roc_rf.std():.4f}")

# Quick training on full data (for reference)
logreg.fit(X, y)
rf.fit(X, y)

print("\n[Note] This supervised benchmark uses disease labels during training and is meant as an upper-bound reference.")